In [10]:
from datetime import datetime
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv("../data/citibike_weather_daily.csv")


In [ ]:
#C1
# Convert ride_date column to numeric value
df['ride_date'] = pd.to_datetime(df['ride_date'])

df['date_numeric'] = df['ride_date'].apply(lambda x: x.timestamp())

df['date_numeric_int'] = df['date_numeric'].astype(int)

print("Original date vs numeric conversion:")
print(df[['ride_date', 'date_numeric', 'date_numeric_int']].head(10))



Original date vs numeric conversion:
   ride_date  date_numeric  date_numeric_int
0 2013-07-01  1.372637e+09        1372636800
1 2013-07-02  1.372723e+09        1372723200
2 2013-07-03  1.372810e+09        1372809600
3 2013-07-04  1.372896e+09        1372896000
4 2013-07-05  1.372982e+09        1372982400
5 2013-07-06  1.373069e+09        1373068800
6 2013-07-07  1.373155e+09        1373155200
7 2013-07-08  1.373242e+09        1373241600
8 2013-07-09  1.373328e+09        1373328000
9 2013-07-10  1.373414e+09        1373414400

Data types:
ride_date           datetime64[us]
date_numeric               float64
date_numeric_int             int64
dtype: object


In [ ]:
# One-hot encode the day_of_week column

# Method 1: Using pd.get_dummies()
day_dummies = pd.get_dummies(df['day_of_week'], prefix='day')

# Add the one-hot encoded columns to the dataframe
df_encoded = pd.concat([df, day_dummies], axis=1)

# Display the results
print("Original day_of_week column:")
print(df[['ride_date', 'day_of_week']].head(10))
print("\nOne-hot encoded columns:")
print(df_encoded[['ride_date', 'day_of_week', 'day_Monday', 'day_Tuesday', 'day_Wednesday', 
                   'day_Thursday', 'day_Friday', 'day_Saturday', 'day_Sunday']].head(10))



Original day_of_week column:
   ride_date day_of_week
0 2013-07-01      Monday
1 2013-07-02     Tuesday
2 2013-07-03   Wednesday
3 2013-07-04    Thursday
4 2013-07-05      Friday
5 2013-07-06    Saturday
6 2013-07-07      Sunday
7 2013-07-08      Monday
8 2013-07-09     Tuesday
9 2013-07-10   Wednesday

One-hot encoded columns:
   ride_date day_of_week  day_Monday  day_Tuesday  day_Wednesday  \
0 2013-07-01      Monday        True        False          False   
1 2013-07-02     Tuesday       False         True          False   
2 2013-07-03   Wednesday       False        False           True   
3 2013-07-04    Thursday       False        False          False   
4 2013-07-05      Friday       False        False          False   
5 2013-07-06    Saturday       False        False          False   
6 2013-07-07      Sunday       False        False          False   
7 2013-07-08      Monday        True        False          False   
8 2013-07-09     Tuesday       False         True         

In [11]:


# Replace sentinel values with NaN
# Temperature columns
df['temp_f'] = df['temp_f'].replace(9999.9, np.nan)
df['max_temp_f'] = df['max_temp_f'].replace(9999.9, np.nan)
df['min_temp_f'] = df['min_temp_f'].replace(9999.9, np.nan)

# Wind column
df['wind_speed_knots'] = df['wind_speed_knots'].replace(999.9, np.nan)

# Precipitation column
df['precip_in'] = df['precip_in'].replace(99.99, np.nan)

# Check for NaN values after replacement
print("\nNaN counts after cleaning:")
print(df[['temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in']].isna().sum())

# Display summary statistics to verify
print("\nSummary statistics after cleaning:")
print(df[['temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in']].describe())


NaN counts after cleaning:
temp_f              0
max_temp_f          0
min_temp_f          0
wind_speed_knots    0
precip_in           1
dtype: int64

Summary statistics after cleaning:
            temp_f   max_temp_f   min_temp_f  wind_speed_knots    precip_in
count  1610.000000  1610.000000  1610.000000       1610.000000  1609.000000
mean     57.368509    67.317081    49.295901          9.013913     0.116364
std      17.876211    18.349408    17.832318          3.241211     0.313778
min       8.300000    18.000000     1.000000          2.300000     0.000000
25%      43.625000    53.700000    35.700000          6.700000     0.000000
50%      59.300000    70.000000    51.100000          8.500000     0.000000
75%      73.500000    82.900000    64.900000         10.800000     0.060000
max      92.500000   100.900000    82.900000         22.400000     4.880000


In [ ]:
# C4: Build a Trend Feature
# Create features to capture system growth over time

# Ensure ride_date is datetime (already done earlier, but good to confirm)
df['ride_date'] = pd.to_datetime(df['ride_date'])

# Option 1: Extract year as a feature
df['year'] = df['ride_date'].dt.year

# Option 2: Create days-since-launch index
# Find the first date in the dataset (launch date)
launch_date = df['ride_date'].min()
print(f"Citi Bike launch date (first date in dataset): {launch_date}")

# Calculate days since launch
df['days_since_launch'] = (df['ride_date'] - launch_date).dt.days

# Display the new features
print("\nTrend features created:")
print(df[['ride_date', 'year', 'days_since_launch', 'num_rides']].head(15))

# Show the relationship between these features
print(f"\nData spans from {df['ride_date'].min()} to {df['ride_date'].max()}")
print(f"Years covered: {sorted(df['year'].unique())}")
print(f"Days since launch ranges from {df['days_since_launch'].min()} to {df['days_since_launch'].max()}")

# Verify correlation with ridership growth
print(f"\nCorrelation with num_rides:")
print(f"  year: {df['year'].corr(df['num_rides']):.4f}")
print(f"  days_since_launch: {df['days_since_launch'].corr(df['num_rides']):.4f}")

In [ ]:
# Save the cleaned and processed dataframe to a new CSV file
# Including ALL transformations: date conversion, NaN replacement, one-hot encoding, trend features

# Merge all one-hot encoded columns into the main dataframe
df_final = pd.concat([df, day_dummies], axis=1)

print("Summary of processing steps applied:")
print("1. Converted ride_date to datetime")
print("2. Created date_numeric and date_numeric_int features")
print("3. Replaced sentinel values with NaN (9999.9, 999.9, 99.99)")
print("4. Created year and days_since_launch trend features")
print("5. One-hot encoded day_of_week (7 new binary columns)")

print(f"\nFinal dataframe shape: {df_final.shape}")
print(f"\nAll columns in cleaned dataset:")
print(list(df_final.columns))


# Save to CSV
output_path = "../data/citibike_weather_daily_clean.csv"
df_final.to_csv(output_path, index=False)

print(f"\n✓ Cleaned data saved to: {output_path}")
print(f"✓ Total columns: {df_final.shape[1]}")
print(f"✓ Total rows: {df_final.shape[0]}")

Summary of processing steps applied:
1. Converted ride_date to datetime
2. Created date_numeric and date_numeric_int features
3. Replaced sentinel values with NaN (9999.9, 999.9, 99.99)
4. Created year and days_since_launch trend features
5. One-hot encoded day_of_week (7 new binary columns)

Final dataframe shape: (1610, 19)

All columns in cleaned dataset:
['ride_date', 'num_rides', 'avg_duration_min', 'temp_f', 'max_temp_f', 'min_temp_f', 'wind_speed_knots', 'precip_in', 'day_of_week', 'month', 'year', 'days_since_launch', 'day_Friday', 'day_Monday', 'day_Saturday', 'day_Sunday', 'day_Thursday', 'day_Tuesday', 'day_Wednesday']

Missing values (NaN) by column:
precip_in    1
dtype: int64

✓ Cleaned data saved to: ../data/citibike_weather_daily_clean.csv
✓ Total columns: 19
✓ Total rows: 1610
